In [ ]:
#!/usr/bin/env python3
""" File: /home/lq53/mir_repos/BBOP/random_tests/25may_mini/
    miniscope_roi_pipeline_with_learning_global_th_hungarian.py

An ROI‐mapping pipeline with built‐in feedback learning,
fixed global thresholds, unique 1:1 mapping via Hungarian algorithm,
and cluster-based merging of similar ROIs across sessions.

Outputs:
  - Per-iteration metrics CSVs
  - Combined overlap map of all matched ROI masks across sessions
  - A one-to-one general→raw mapping CSV for downstream signal extraction
  - Session‐level summary overlays (Hungarian)
  - ROI side-by-side overlays (raw vs mapped)
  - Combined union masks per general ROI
"""
import os, glob
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import shift as ndi_shift
from scipy.optimize import linear_sum_assignment
from skimage.morphology import binary_dilation, disk
from skimage.feature import match_template
from skimage.measure import regionprops

# — Configuration —————————————————————————————————————
session_dirs = [
    '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_12/20241001PMCRE2mini_15_22',
    '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_12/20241001PMCRE2mini_13_57',
    '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_12/20241001PMCRE2mini_13_44',
    '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_12/20241001PMCRE2mini_15_35',
]
map_csv = '/home/lq53/mir_repos/BBOP/random_tests/25may_mini/mini_nc_mapping_250618.csv'

PAD            = 10     # px padding around ROI template
DILATE_R       = 5      # px to dilate reference masks
ALPHA          = 0.7    # weight for NCC vs IoU
DIST_TH        = 6.5    # px threshold for centroid distance
GLOBAL_COMB_TH = 0.69   # combined-score threshold
ITER           = 2      # learning iterations
MIN_IOU_TH     = 0.1    # minimum IoU for final mapping
MERGE_IOU_TH   = 0.4    # IoU threshold to cluster/merge similar masks

# — Output directories ———————————————————————————————————
out_base     = os.path.expanduser('/data/big_rim/mir_data/roi_alignments_tests/roi_learning_results_comb2')
metrics_dir  = os.path.join(out_base, 'metrics')
figures_dir  = os.path.join(out_base, 'figures')
overlays_sum = os.path.join(out_base, 'overlays_hungarian')
overlays_sb  = os.path.join(out_base, 'overlays_side_by_side')
for d in (out_base, metrics_dir, figures_dir, overlays_sum, overlays_sb):
    os.makedirs(d, exist_ok=True)

# — Helper functions —————————————————————————————————————
def get_mapped_rec_path(sd):
    txts = glob.glob(os.path.join(sd, '*.txt'))
    if not txts:
        raise FileNotFoundError(f"No mapping .txt in {sd}")
    return open(txts[0], 'r').read().strip()

def load_minian_data(nc_path, ts_path):
    ds = xr.open_dataset(nc_path).load()
    ts = pd.read_csv(ts_path)
    return ds, ts['Time Stamp (ms)'].values

def extract_masks(ds, dilation_radius=0):
    A = ds['A']
    arr = A.values if hasattr(A, 'values') else A.toarray()
    n, H, W = arr.shape[0], *ds['max_proj'].shape
    masks = arr.reshape(n, H, W) > 0
    if dilation_radius > 0:
        se = disk(dilation_radius)
        masks = np.stack([binary_dilation(m, se) for m in masks])
    return masks

def compute_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter/union if union>0 else 0

# — Load reference session ———————————————————————————————————
df_map   = pd.read_csv(map_csv, names=['rec_path','nc_key'])
rec0_sd  = session_dirs[0]
rec0     = get_mapped_rec_path(rec0_sd) + '/My_V4_Miniscope'
key0     = df_map.loc[df_map.rec_path==rec0, 'nc_key'].iat[0]
nc0      = os.path.join(rec0, f'minian_dataset_{key0}.nc')
ts0      = os.path.join(rec0, 'timeStamps.csv')
ds0, _   = load_minian_data(nc0, ts0)
proj0    = ds0['max_proj'].values

general_masks0 = extract_masks(ds0, dilation_radius=DILATE_R)

# — Load all session raw masks —————————————————————————————————
projections = [proj0]
session_raw_masks = []
for sd in session_dirs[1:]:
    rec_sd = get_mapped_rec_path(sd) + '/My_V4_Miniscope'
    key   = df_map.loc[df_map.rec_path==rec_sd, 'nc_key'].iat[0]
    ds, _ = load_minian_data(
        os.path.join(rec_sd, f'minian_dataset_{key}.nc'),
        os.path.join(rec_sd, 'timeStamps.csv')
    )
    projections.append(ds['max_proj'].values)
    session_raw_masks.append(extract_masks(ds, dilation_radius=0))

# — Combine reference + raw masks into a single list —————————————————
general_masks = list(general_masks0)
mask_session_idx_orig = [0]*len(general_masks0)
for sess_i, raw in enumerate(session_raw_masks, start=1):
    for m in raw:
        general_masks.append(m)
        mask_session_idx_orig.append(sess_i)

# — Cluster-based merging via IoU adjacency —————————————————————
N = len(general_masks)
adj = [set() for _ in range(N)]
for i in range(N):
    for j in range(i+1, N):
        if compute_iou(general_masks[i], general_masks[j]) >= MERGE_IOU_TH:
            adj[i].add(j); adj[j].add(i)
visited = set()
merged_masks = []
merged_sessions = []
for i in range(N):
    if i in visited: continue
    stack = [i]
    comp = []
    while stack:
        v = stack.pop()
        if v in visited: continue
        visited.add(v)
        comp.append(v)
        for w in adj[v]:
            if w not in visited:
                stack.append(w)
    # union cluster
    union = general_masks[comp[0]].copy()
    sess_rep = mask_session_idx_orig[comp[0]]
    for k in comp[1:]:
        union = np.logical_or(union, general_masks[k])
    merged_masks.append(union)
    merged_sessions.append(sess_rep)
# overwrite generalized set
general_masks = merged_masks
mask_session_index = merged_sessions

# — Build templates + origins for each general mask ——————————————————
templates = []
origins   = []
for m, sess_i in zip(general_masks, mask_session_index):
    proj = projections[sess_i]
    minr, minc, maxr, maxc = regionprops(m.astype(int))[0].bbox
    minr = max(0, minr-PAD); minc = max(0, minc-PAD)
    maxr = min(proj.shape[0], maxr+PAD); maxc = min(proj.shape[1], maxc+PAD)
    templates.append(proj[minr:maxr, minc:maxc].astype(float))
    origins.append((minr, minc))

# — Iterative NCC + IoU matching —————————————————————————————————
matched_masks = None
for it in range(1, ITER+1):
    records, new_masks = [], []
    for j, sd in enumerate(session_dirs[1:], start=1):
        proj = projections[j]; this_session = []
        for idx, (tmpl, (origr, origc)) in enumerate(zip(templates, origins)):
            res  = match_template(proj, tmpl)
            mc   = res.max()
            y, x = np.unravel_index(res.argmax(), res.shape)
            dr, dc = y-origr, x-origc
            moved = ndi_shift(general_masks[idx].astype(float), (dr, dc), order=0) > 0.5
            if not moved.any():
                records.append({'roi_idx':idx,'session':sd,'matched':False,'ncc':mc,'iou':np.nan,'dist':np.nan})
                this_session.append(None)
                continue
            cg   = np.array(regionprops(general_masks[idx].astype(int))[0].centroid)
            cm   = np.array(regionprops(moved.astype(int))[0].centroid)
            dist = np.linalg.norm(cm-cg)
            iou  = compute_iou(general_masks[idx], moved)
            records.append({'roi_idx':idx,'session':sd,'matched':True,'ncc':mc,'iou':iou,'dist':dist})
            this_session.append(moved)
        new_masks.append(this_session)
    df_iter = pd.DataFrame(records)
    df_iter['combined'] = ALPHA*df_iter['ncc'] + (1-ALPHA)*df_iter['iou']
    df_iter['good']     = (df_iter['matched'] & (df_iter['dist']<=DIST_TH) & (df_iter['combined']>=GLOBAL_COMB_TH))
    df_iter.to_csv(os.path.join(metrics_dir, f'metrics_{it}.csv'), index=False)
    # refine templates by weighted average of good matches
    updated = []
    for idx, tmpl in enumerate(templates):
        patches, wts = [tmpl], [1.0]
        for _, r in df_iter[(df_iter.roi_idx==idx)&df_iter.good].iterrows():
            res2 = match_template(projections[session_dirs.index(r.session)], tmpl)
            y2, x2 = np.unravel_index(res2.argmax(), res2.shape)
            h, w   = tmpl.shape
            patch  = projections[session_dirs.index(r.session)][y2:y2+h, x2:x2+w].astype(float)
            patches.append(patch); wts.append(r.combined)
        stack = np.stack(patches)
        W     = np.array(wts)[:,None,None]
        updated.append((stack*W).sum(0)/W.sum())
    templates     = updated
    matched_masks = new_masks

# — Combined overlap map —————————————————————————————————————
all_masks = [m for sess in matched_masks for m in sess if m is not None]
if all_masks:
    sm = np.stack(all_masks).sum(axis=0)
    out = os.path.join(figures_dir, 'sum_map_after.png')
    plt.imsave(out, sm, cmap='viridis')
    print(f"Saved overlap map to {out}")

# — Hungarian one-to-one mapping —————————————————————————————————
mapping = []
for j, sd in enumerate(session_dirs[1:], start=1):
    raw  = session_raw_masks[j-1]
    mov  = matched_masks[j-1]
    n_gen, n_raw = len(mov), len(raw)
    C    = np.zeros((n_gen, n_raw))
    for i in range(n_gen):
        for k in range(n_raw):
            C[i,k] = -compute_iou(mov[i], raw[k]) if mov[i] is not None else 0.0
    rows, cols = linear_sum_assignment(C)
    for i,k in zip(rows, cols):
        if mov[i] is None: continue
        val = compute_iou(mov[i], raw[k])
        if val < MIN_IOU_TH: continue
        mapping.append({'session':sd,'general_idx':i,'raw_idx':k,'iou':val})
mp = pd.DataFrame(mapping)
mp.to_csv(os.path.join(figures_dir,'general_to_raw_hungarian.csv'), index=False)
print('Saved general→raw mapping CSV')

# — Session summaries & overlays ——————————————————————————————
for j, sd in enumerate(session_dirs[1:], start=1):
    fig, axs = plt.subplots(1,2,figsize=(12,6))
    axs[0].imshow(projections[j], cmap='gray'); axs[0].axis('off'); axs[0].set_title(f'Session {j} Raw')
    for idx, mask in enumerate(session_raw_masks[j-1]):
        if mask is None: continue
        p = regionprops(mask.astype(int))[0].centroid
        axs[0].contour(mask, levels=[0.5], colors='lime', alpha=0.5)
        axs[0].text(p[1], p[0], str(idx), color='lime', fontsize=8, ha='center')
    axs[1].imshow(projections[j], cmap='gray'); axs[1].axis('off'); axs[1].set_title(f'Mapped ROIs')
    for rec in mapping:
        if rec['session']!=sd: continue
        m = session_raw_masks[j-1][rec['raw_idx']]
        p = regionprops(m.astype(int))[0].centroid
        axs[1].contour(m, levels=[0.5], colors='magenta', alpha=0.5)
        axs[1].text(p[1], p[0], str(rec['general_idx']), color='magenta', fontsize=8, ha='center')
    outf = os.path.join(overlays_sum, f'session{j}_summary.png')
    plt.tight_layout(); plt.savefig(outf, bbox_inches='tight'); plt.close(fig)
    print(f"Saved overlay for session {j}")

# — Combined union masks visualization ————————————————————————————
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(proj0, cmap='gray'); ax.axis('off'); ax.set_title('Combined Union Masks')
for idx, union in enumerate(general_masks):
    p = regionprops(union.astype(int))[0].centroid
    ax.contour(union, levels=[0.5], colors='yellow', alpha=0.6)
    ax.text(p[1], p[0], str(idx), color='yellow', fontsize=8, ha='center')
out = os.path.join(figures_dir, 'combined_union.png')
plt.tight_layout(); plt.savefig(out, bbox_inches='tight'); plt.close(fig)
print(f"Saved combined union to {out}")


In [11]:
#!/usr/bin/env python3
""" File: /home/lq53/mir_repos/BBOP/random_tests/25may_mini/
    miniscope_roi_pipeline_with_learning_global_th_hungarian.py

An ROI‑mapping pipeline with built‑in feedback learning,
fixed global thresholds, unique 1:1 mapping via Hungarian algorithm,
and cluster‑based merging of similar ROIs across sessions.

Outputs:
  - Per‑iteration metrics CSVs
  - Combined overlap map of all matched ROI masks across sessions
  - A one‑to‑one general→raw mapping CSV for downstream signal extraction
  - Session‑level summary overlays (Hungarian)
  - ROI side-by-side overlays (raw vs mapped)
  - Combined union masks per general ROI
"""
import os, glob
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import shift as ndi_shift
from scipy.optimize import linear_sum_assignment
from skimage.morphology import binary_dilation, disk
from skimage.feature import match_template
from skimage.measure import regionprops
import json
from datetime import datetime

# — Configuration —————————————————————————————————————
session_dirs = [
    '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_12/20241001PMCRE2mini_15_22',
    '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_12/20241001PMCRE2mini_13_57',
    '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_12/20241001PMCRE2mini_13_44',
    '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_12/20241001PMCRE2mini_15_35',
]
map_csv = '/home/lq53/mir_repos/BBOP/random_tests/25may_mini/mini_nc_mapping_250618.csv'

PAD            = 10     # px padding around ROI template
DILATE_R       = 6 #5      # px to dilate reference masks
ALPHA          = 0.7    # weight for NCC vs IoU
DIST_TH        = 6.5    # px threshold for centroid distance
GLOBAL_COMB_TH = 0.69   # combined-score threshold
ITER           = 4     # learning iterations
MIN_IOU_TH     = 0.2 #0.1    # minimum IoU for final mapping
MERGE_IOU_TH   = 0.3 #0.4    # IoU threshold to cluster/merge similar masks


# — Save this run’s configuration for later comparison —————————————————
config = {
    'PAD': PAD,
    'DILATE_R': DILATE_R,
    'ALPHA': ALPHA,
    'DIST_TH': DIST_TH,
    'GLOBAL_COMB_TH': GLOBAL_COMB_TH,
    'ITER': ITER,
    'MIN_IOU_TH': MIN_IOU_TH,
    'MERGE_IOU_TH': MERGE_IOU_TH,
}




# — Output directories ———————————————————————————————————
# out_base     = os.path.expanduser('/data/big_rim/mir_data/roi_alignments_tests/250724_roi_learning_results_comb2_exclude')
now = datetime.now()
timestamp = now.strftime("%y%m%d_%H%M")  # e.g. 250719_1605

# Output path with timestamp in folder name
out_base = os.path.expanduser(
    f"/data/big_rim/mir_data/roi_alignments_tests/roi_learning_{timestamp}"
)


metrics_dir  = os.path.join(out_base, 'metrics')
figures_dir  = os.path.join(out_base, 'figures')
overlays_sum = os.path.join(out_base, 'overlays_hungarian')
overlays_sb  = os.path.join(out_base, 'overlays_side_by_side')
for d in (out_base, metrics_dir, figures_dir, overlays_sum, overlays_sb):
    os.makedirs(d, exist_ok=True)

# — Helper functions —————————————————————————————————————
def get_mapped_rec_path(sd):
    txts = glob.glob(os.path.join(sd, '*.txt'))
    if not txts:
        raise FileNotFoundError(f"No mapping .txt in {sd}")
    return open(txts[0], 'r').read().strip()

def load_minian_data(nc_path, ts_path):
    ds = xr.open_dataset(nc_path).load()
    ts = pd.read_csv(ts_path)
    return ds, ts['Time Stamp (ms)'].values

def extract_masks(ds, dilation_radius=0):
    A = ds['A']
    arr = A.values if hasattr(A, 'values') else A.toarray()
    n, H, W = arr.shape[0], *ds['max_proj'].shape
    masks = arr.reshape(n, H, W) > 0
    if dilation_radius > 0:
        se = disk(dilation_radius)
        masks = np.stack([binary_dilation(m, se) for m in masks])
    return masks

def compute_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter/union if union > 0 else 0

# — Load reference session ———————————————————————————————————
df_map   = pd.read_csv(map_csv, names=['rec_path','nc_key'])
rec0_sd  = session_dirs[0]
rec0     = get_mapped_rec_path(rec0_sd) + '/My_V4_Miniscope'
key0     = df_map.loc[df_map.rec_path==rec0, 'nc_key'].iat[0]
nc0      = os.path.join(rec0, f'minian_dataset_{key0}.nc')
ts0      = os.path.join(rec0, 'timeStamps.csv')
ds0, _   = load_minian_data(nc0, ts0)
proj0    = ds0['max_proj'].values

general_masks0 = extract_masks(ds0, dilation_radius=DILATE_R)

# — Load all session raw masks —————————————————————————————————
projections = [proj0]
session_raw_masks = []
for sd in session_dirs[1:]:
    rec_sd = get_mapped_rec_path(sd) + '/My_V4_Miniscope'
    key    = df_map.loc[df_map.rec_path==rec_sd, 'nc_key'].iat[0]
    ds, _  = load_minian_data(
        os.path.join(rec_sd, f'minian_dataset_{key}.nc'),
        os.path.join(rec_sd, 'timeStamps.csv')
    )
    projections.append(ds['max_proj'].values)
    session_raw_masks.append(extract_masks(ds, dilation_radius=0))

# — Combine reference + raw masks into a single list —————————————————
general_masks         = list(general_masks0)
mask_session_idx_orig = [0] * len(general_masks0)
for sess_i, raw in enumerate(session_raw_masks, start=1):
    for m in raw:
        general_masks.append(m)
        mask_session_idx_orig.append(sess_i)

# — Cluster-based merging via IoU adjacency —————————————————————
N = len(general_masks)
adj = [set() for _ in range(N)]
for i in range(N):
    for j in range(i+1, N):
        if compute_iou(general_masks[i], general_masks[j]) >= MERGE_IOU_TH:
            adj[i].add(j)
            adj[j].add(i)
visited = set()
merged_masks    = []
merged_sessions = []
for i in range(N):
    if i in visited:
        continue
    stack = [i]
    comp  = []
    while stack:
        v = stack.pop()
        if v in visited:
            continue
        visited.add(v)
        comp.append(v)
        for w in adj[v]:
            if w not in visited:
                stack.append(w)
    union    = general_masks[comp[0]].copy()
    sess_rep = mask_session_idx_orig[comp[0]]
    for k in comp[1:]:
        union = np.logical_or(union, general_masks[k])
    merged_masks.append(union)
    merged_sessions.append(sess_rep)
# overwrite generalized set
general_masks     = merged_masks
mask_session_index = merged_sessions

# — Build templates + origins for each general mask ——————————————————
templates = []
origins   = []
for m, sess_i in zip(general_masks, mask_session_index):
    proj = projections[sess_i]
    minr, minc, maxr, maxc = regionprops(m.astype(int))[0].bbox
    minr = max(0, minr - PAD)
    minc = max(0, minc - PAD)
    maxr = min(proj.shape[0], maxr + PAD)
    maxc = min(proj.shape[1], maxc + PAD)
    templates.append(proj[minr:maxr, minc:maxc].astype(float))
    origins.append((minr, minc))

# — Iterative NCC + IoU matching —————————————————————————————————
matched_masks = None
df_iter       = None
for it in range(1, ITER + 1):
    records, new_masks = [], []
    for j, sd in enumerate(session_dirs[1:], start=1):
        proj = projections[j]
        this_session = []
        for idx, (tmpl, (origr, origc)) in enumerate(zip(templates, origins)):
            res = match_template(proj, tmpl)
            mc   = res.max()
            y, x = np.unravel_index(res.argmax(), res.shape)
            dr, dc = y - origr, x - origc
            moved = ndi_shift(general_masks[idx].astype(float), (dr, dc), order=0) > 0.5
            if not moved.any():
                records.append({'roi_idx': idx, 'session': sd, 'matched': False, 'ncc': mc, 'iou': np.nan, 'dist': np.nan})
                this_session.append(None)
                continue
            cg = np.array(regionprops(general_masks[idx].astype(int))[0].centroid)
            cm = np.array(regionprops(moved.astype(int))[0].centroid)
            dist = np.linalg.norm(cm - cg)
            iou  = compute_iou(general_masks[idx], moved)
            records.append({'roi_idx': idx, 'session': sd, 'matched': True, 'ncc': mc, 'iou': iou, 'dist': dist})
            this_session.append(moved)
        new_masks.append(this_session)
    df_iter = pd.DataFrame(records)
    df_iter['combined'] = ALPHA * df_iter['ncc'] + (1 - ALPHA) * df_iter['iou']
    df_iter['good']     = (df_iter['matched'] & (df_iter['dist'] <= DIST_TH) & (df_iter['combined'] >= GLOBAL_COMB_TH))
    df_iter.to_csv(os.path.join(metrics_dir, f'metrics_{it}.csv'), index=False)
    # refine templates by weighted average of good matches
    updated = []
    for idx, tmpl in enumerate(templates):
        patches, wts = [tmpl], [1.0]
        for _, r in df_iter[(df_iter.roi_idx == idx) & df_iter.good].iterrows():
            res2 = match_template(projections[session_dirs.index(r.session)], tmpl)
            y2, x2 = np.unravel_index(res2.argmax(), res2.shape)
            h, w = tmpl.shape
            patch = projections[session_dirs.index(r.session)][y2:y2+h, x2:x2+w].astype(float)
            patches.append(patch)
            wts.append(r.combined)
        stack = np.stack(patches)
        W = np.array(wts)[:, None, None]
        updated.append((stack * W).sum(0) / W.sum())
    templates     = updated
    matched_masks = new_masks

# — Drop wrong ROIs —————————————————————————————————————
if df_iter is not None:
    good_idx = df_iter.groupby('roi_idx')['good'].all().loc[lambda x: x].index.tolist()
    general_masks = [general_masks[i] for i in good_idx]
    templates     = [templates[i] for i in good_idx]
    origins       = [origins[i] for i in good_idx]
    matched_masks = [[sess[i] for i in good_idx] for sess in matched_masks]

# — Combined overlap map —————————————————————————————————————
all_masks = [m for sess in matched_masks for m in sess if m is not None]
if all_masks:
    sm  = np.stack(all_masks).sum(axis=0)
    out = os.path.join(figures_dir, 'sum_map_after.png')
    plt.imsave(out, sm, cmap='viridis')
    print(f"Saved overlap map to {out}")

# — Hungarian one-to-one mapping —————————————————————————————————
mapping = []
for j, sd in enumerate(session_dirs[1:], start=1):
    raw  = session_raw_masks[j-1]
    mov  = matched_masks[j-1]
    n_gen, n_raw = len(mov), len(raw)
    C    = np.zeros((n_gen, n_raw))
    for i in range(n_gen):
        for k in range(n_raw):
            C[i, k] = -compute_iou(mov[i], raw[k]) if mov[i] is not None else 0.0
    rows, cols = linear_sum_assignment(C)
    for i, k in zip(rows, cols):
        if mov[i] is None:
            continue
        val = compute_iou(mov[i], raw[k])
        if val < MIN_IOU_TH:
            continue
        mapping.append({'session': sd, 'general_idx': i, 'raw_idx': k, 'iou': val})
mp = pd.DataFrame(mapping)
mp.to_csv(os.path.join(figures_dir, 'general_to_raw_hungarian.csv'), index=False)
print('Saved general→raw mapping CSV')

# — Session summaries & overlays ——————————————————————————————
for j, sd in enumerate(session_dirs[1:], start=1):
    fig, axs = plt.subplots(1,2,figsize=(12,6))
    axs[0].imshow(projections[j], cmap='gray'); axs[0].axis('off'); axs[0].set_title(f'Session {j} Raw')
    for idx, mask in enumerate(session_raw_masks[j-1]):
        if mask is None: continue
        p = regionprops(mask.astype(int))[0].centroid
        axs[0].contour(mask, levels=[0.5], colors='lime', alpha=0.5)
        axs[0].text(p[1], p[0], str(idx), color='lime', fontsize=8, ha='center')
    axs[1].imshow(projections[j], cmap='gray'); axs[1].axis('off'); axs[1].set_title(f'Mapped ROIs')
    for rec in mapping:
        if rec['session']!=sd: continue
        m = session_raw_masks[j-1][rec['raw_idx']]
        p = regionprops(m.astype(int))[0].centroid
        axs[1].contour(m, levels=[0.5], colors='magenta', alpha=0.5)
        axs[1].text(p[1], p[0], str(rec['general_idx']), color='magenta', fontsize=8, ha='center')
    outf = os.path.join(overlays_sum, f'session{j}_summary.png')
    plt.tight_layout(); plt.savefig(outf, bbox_inches='tight'); plt.close(fig)
    print(f"Saved overlay for session {j}")

# — Combined union masks visualization ————————————————————————————
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(proj0, cmap='gray'); ax.axis('off'); ax.set_title('Combined Union Masks')
for idx, union in enumerate(general_masks):
    p = regionprops(union.astype(int))[0].centroid
    ax.contour(union, levels=[0.5], colors='yellow', alpha=0.6)
    ax.text(p[1], p[0], str(idx), color='yellow', fontsize=8, ha='center')
out = os.path.join(figures_dir, 'combined_union.png')
plt.tight_layout(); plt.savefig(out, bbox_inches='tight'); plt.close(fig)
print(f"Saved combined union to {out}")

with open(os.path.join(out_base, 'run_config.json'), 'w') as cf:
    json.dump(config, cf, indent=2)

Saved overlap map to /data/big_rim/mir_data/roi_alignments_tests/roi_learning_250724_1544/figures/sum_map_after.png
Saved general→raw mapping CSV
Saved overlay for session 1
Saved overlay for session 2
Saved overlay for session 3
Saved combined union to /data/big_rim/mir_data/roi_alignments_tests/roi_learning_250724_1544/figures/combined_union.png
